# ComicAnalizer - OCR Complementario

Notebook separado para probar PaddleOCR contra resultados Magi ya generados. Usa como entrada `magi_clean_full.zip` y un ZIP de resultados Magi/Colab, o una carpeta Magi existente en el runtime.

In [ ]:
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!pip -q install "paddleocr==3.3.3" "paddlepaddle==3.2.0"

## Subir entradas

Sube el ZIP del dataset limpio y el ZIP de resultados Magi. Para `nekkorarekko`, usa:

- `magi_nekkorarekko_clean.zip`
- `nekkorarekko_clean_magi_ocr_outputs.zip` o el ZIP Magi que descargaste desde el notebook principal


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
print('Subidos:', list(uploaded))

DATASET_ZIP = next(name for name in uploaded if name.startswith('magi_') and 'clean' in name and name.endswith('.zip'))
PACKAGE_NAME = DATASET_ZIP.replace('.zip', '')
MAGI_RESULTS_ZIP = next((name for name in uploaded if name != DATASET_ZIP and name.endswith('.zip')), '')

!rm -rf /content/magi_sample /content/magi_results_input
!mkdir -p /content/magi_sample /content/magi_results_input
!unzip -q -o "$DATASET_ZIP" -d /content/magi_sample
if MAGI_RESULTS_ZIP:
    !unzip -q -o "$MAGI_RESULTS_ZIP" -d /content/magi_results_input

print('DATASET_ZIP:', DATASET_ZIP)
print('PACKAGE_NAME:', PACKAGE_NAME)
print('MAGI_RESULTS_ZIP:', MAGI_RESULTS_ZIP)
!find /content/magi_sample -maxdepth 5 -type d | head -30
!find /content/magi_results_input -maxdepth 7 -name magi_results.json -o -name metrics.json | head -20


In [ ]:
RUN_NAME = 'nekkorarekko_ocr_comparison'
DATASET_NAME = 'test_1_clean'
COMIC_ID = 'nekkorarekko'  # usa '' para no filtrar
RUN_ROOT = f'outputs/runs/{RUN_NAME}'
ANALYSIS_OUTPUT = f'{RUN_ROOT}/analysis/magi_analysis_report.json'
OCR_OUTPUT = f'{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json'
OCR_VISUALS = f'{RUN_ROOT}/visuals/ocr_boxes'

from pathlib import Path

standard_magi_candidates = sorted(Path('/content/magi_results_input').rglob('magi_results.json'))
MAGI_INPUT = str(standard_magi_candidates[0].parent) if standard_magi_candidates else ''
IMAGE_ROOT = f'/content/magi_sample/{PACKAGE_NAME}/by_comic'
OCR_LIMIT = 12
print('MAGI_INPUT:', MAGI_INPUT)
print('IMAGE_ROOT:', IMAGE_ROOT)


In [ ]:
ocr_comic_filter = f'--comic-id {COMIC_ID}' if COMIC_ID else ''

!python -m tools.analyze_magi_results   --input $MAGI_INPUT   --output $ANALYSIS_OUTPUT   --top-n 20

!python -m tools.compare_magi_paddleocr   --magi-input $MAGI_INPUT   --image-root $IMAGE_ROOT   --dataset-name $DATASET_NAME   --selection random   --limit $OCR_LIMIT   --seed 42   --lang en   --visual-output-dir $OCR_VISUALS   --output $OCR_OUTPUT   $ocr_comic_filter


In [ ]:
import json
from pathlib import Path

ocr_report = json.loads(Path(OCR_OUTPUT).read_text())
print(json.dumps(ocr_report['summary'], indent=2, ensure_ascii=False))
for item in ocr_report['comparisons']:
    print(item['comic_id'], item['file_name'], 'Magi=', item['magi_text_regions'], 'Paddle=', item['paddle_text_blocks'], 'match=', item['matched_regions'], 't=', round(item['paddle_elapsed_seconds'], 2))

In [ ]:
from google.colab import files

zip_out = f'{RUN_NAME}_outputs.zip'
!zip -qr "$zip_out" $RUN_ROOT
files.download(zip_out)
